**Objetivo:** limpar, tipar e padronizar os dados da Bronze (que **não é alterada**), com nomes de colunas em português.

**Princípios adotados**
- **Conversões seguras:** nenhum valor sujo pode derrubar o pipeline. Todo cast só acontece depois de validar o formato por regex; o que não converte vira `NULL` (compatível com modo ANSI do Databricks).
- **Deduplicação pela ingestão mais recente:** como a Bronze é *append*, cada reexecução repete as linhas; mantemos a versão de maior `ingestion_datetime` (desempate: registro mais completo).
- **Silver idempotente:** cada tabela é reescrita por completo (`overwrite`) a cada execução.

**Ordem:** `tb_cotacao_dolar` é construída primeiro, pois o financeiro depende da cotação.

In [0]:
from datetime import date

from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "workspace", "Catálogo (Unity Catalog)")
catalog = dbutils.widgets.get("catalog").strip()

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Se um formato de data não puder ser interpretado, o Spark deve devolver NULL
# (e não lançar SparkUpgradeException). Necessário para o parse multi-formato.
try:
    spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")
except Exception as e:
    print("[aviso] não foi possível ajustar timeParserPolicy:", e)

In [0]:
# Textos que representam "ausência de dado" nas colunas financeiras
AUSENTES = ["unknown", "não informado", "nao informado", "n/a", "na", "null", "none", "nan", "-", "--", "?", ""]

SCI_RE = r"^-?\d+(\.\d+)?[eE][+-]?(0?[0-9]|1[0-9])$"   # notação científica (ex.: 1.6e8)
NUM_RE = r"^-?\d+(\.\d+)?$"                          # número canônico (ponto decimal)


def require_columns(df, colunas, nome):
    """Erro claro se a Bronze não tiver as colunas esperadas (evita falha obscura mais adiante)."""
    faltando = [c for c in colunas if c not in df.columns]
    if faltando:
        raise ValueError(f"{nome}: colunas ausentes {faltando}. Colunas encontradas: {df.columns}")


def write_silver(df, tabela):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela))
    print(f"[ok] {tabela}: {spark.table(tabela).count():,} linhas")


def blank_to_null(c):
    """trim + string vazia (ou só espaços) -> NULL."""
    t = F.trim(c.cast("string"))
    return F.when(t == "", F.lit(None)).otherwise(t)


def prepare_id(df, col="id"):
    """Normaliza a chave natural: trim, remove sufixo '.0' e descarta ids nulos/vazios (não há como ligar ao filme)."""
    limpo = F.regexp_replace(F.trim(F.col(col).cast("string")), r"\.0+$", "")
    return df.withColumn(col, F.when(limpo == "", F.lit(None)).otherwise(limpo)).filter(F.col(col).isNotNull())


def latest_version(df, key_col="id"):
    """
    Deduplicação: mantém 1 linha por chave, a de maior ingestion_datetime.
    Desempate (mesma ingestão): o registro mais completo (mais campos preenchidos).
    """
    cols = [c for c in df.columns if c not in (key_col, "ingestion_datetime")]
    completude = sum(
        (F.when(F.trim(F.col(c).cast("string")) != "", 1).otherwise(0) for c in cols),
        F.lit(0),
    )
    w = Window.partitionBy(key_col).orderBy(F.col("ingestion_datetime").desc(), completude.desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


def clean_number(c, single_sep_is_decimal=True, allow_currency=False):
    """
    Higieniza um texto numérico e devolve uma STRING canônica ('1234.56') ou NULL.

    - Remove símbolos de moeda (se allow_currency), espaços e pontuação de milhar.
    - Se houver letras/texto (ex.: 'Unknown', lixo de Column Shift) -> NULL.
    - Separadores: se houver '.' e ',' juntos, o ÚLTIMO é o decimal ('1.234,56' e '1,234.56').
      Separador repetido ('1.000.000' / '1,000,000') é milhar.
      Separador único é ambíguo:
        * single_sep_is_decimal=True  (popularidade, notas): tratado como decimal ('12,5' -> 12.5).
        * single_sep_is_decimal=False (dinheiro, contagens): '1,234' / '1.234' (1-3 dígitos + 3) é milhar; demais casos, decimal.
    """
    raw = F.trim(c.cast("string"))
    if allow_currency:
        raw = F.trim(F.regexp_replace(raw, r"(?i)(US\$|R\$|USD|BRL|\$|€|£)", ""))

    is_sci = raw.rlike(SCI_RE)
    has_letters = raw.rlike(r"\p{L}")

    s = F.regexp_replace(raw, r"[^0-9,.\-]", "")
    n_dot = F.length(s) - F.length(F.regexp_replace(s, r"\.", ""))
    n_com = F.length(s) - F.length(F.regexp_replace(s, ",", ""))
    rev = F.reverse(s)
    dot_is_last = F.instr(rev, ".") < F.instr(rev, ",")   # só é usado quando ambos existem

    no_dots = F.regexp_replace(s, r"\.", "")
    no_coms = F.regexp_replace(s, ",", "")
    dec_com = F.regexp_replace(no_dots, ",", ".")

    if single_sep_is_decimal:
        single_com = F.regexp_replace(s, ",", ".")
        single_dot = s
    else:
        single_com = F.when(s.rlike(r"^-?[1-9]\d{0,2},\d{3}$"), no_coms).otherwise(F.regexp_replace(s, ",", "."))
        single_dot = F.when(s.rlike(r"^-?[1-9]\d{0,2}\.\d{3}$"), no_dots).otherwise(s)

    normalizado = (
        F.when((n_dot > 0) & (n_com > 0), F.when(dot_is_last, no_coms).otherwise(dec_com))
        .when(n_com > 1, no_coms)
        .when(n_com == 1, single_com)
        .when(n_dot > 1, no_dots)
        .when(n_dot == 1, single_dot)
        .otherwise(s)
    )

    sci_val = raw.cast("double").cast("decimal(38,10)").cast("string")   # protegido pelo is_sci
    return (
        F.when(is_sci, sci_val)
        .when(has_letters, F.lit(None))
        .when(normalizado.rlike(NUM_RE), normalizado)
    )


def to_decimal(c, precision=18, scale=2, single_sep_is_decimal=False, allow_currency=True, max_int_digits=13):
    """Texto -> DECIMAL seguro. Valores absurdos (> 13 dígitos inteiros) viram NULL para não estourar o tipo."""
    s = clean_number(c, single_sep_is_decimal, allow_currency)
    ok = s.rlike(r"^-?\d{1," + str(max_int_digits) + r"}(\.\d+)?$")
    return F.when(ok, s.cast(f"decimal({precision},{scale})"))


def to_double(c, single_sep_is_decimal=True):
    """Texto -> DOUBLE seguro (texto/lixo -> NULL)."""
    s = clean_number(c, single_sep_is_decimal)
    ok = s.rlike(r"^-?\d{1,15}(\.\d+)?$")
    return F.when(ok, s.cast("double"))


def to_int(c, single_sep_is_decimal=False):
    """Texto -> INT seguro. Fração diferente de zero ou > 9 dígitos -> NULL."""
    s = clean_number(c, single_sep_is_decimal)
    ok = s.rlike(r"^-?\d{1,9}(\.0+)?$")
    return F.when(ok, F.regexp_replace(s, r"\.0+$", "").cast("int"))

Regras: 1 cotação por dia (o último boletim do dia) → calendário contínuo do primeiro dia da série até hoje → dias sem cotação
(finais de semana/feriados) recebem o valor do último dia útil anterior (**forward fill**).
A coluna `preenchida_forward_fill` deixa explícito quais dias foram preenchidos.

In [0]:
cot = spark.table("bronze.tb_cotacao_dolar")
require_columns(cot, ["dataHoraCotacao", "cotacaoCompra"], "bronze.tb_cotacao_dolar")

# A API devolve 'AAAA-MM-DD hh:mm:ss.fff': usamos só a parte da data (validada por regex antes do parse)
data_txt = F.substring(F.trim(F.col("dataHoraCotacao").cast("string")), 1, 10)

cot_limpa = (
    cot.withColumn(
        "data_cotacao",
        F.when(
            data_txt.rlike(r"^\d{4}-\d{2}-\d{2}$"),
            F.try_to_timestamp(data_txt, F.lit("yyyy-MM-dd")).cast("date"),
        ),
    )
    .withColumn("cotacao", F.col("cotacaoCompra").cast("double"))
    .filter(F.col("data_cotacao").isNotNull() & (F.col("cotacao") > 0))
)

# Um valor por dia: o boletim mais tardio (e, em empate, a ingestão mais recente)
w_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc(), F.col("ingestion_datetime").desc())
dias_uteis = (
    cot_limpa.withColumn("_rn", F.row_number().over(w_dia))
    .filter("_rn = 1")
    .select("data_cotacao", F.col("cotacao").cast("decimal(18,6)").alias("cotacao_dolar"))
)

limites = dias_uteis.agg(F.min("data_cotacao").alias("ini"), F.max("data_cotacao").alias("fim")).first()
assert limites["ini"] is not None, "Nenhuma cotação válida na Bronze: reexecute o Landing_to_Bronze com uma janela maior."

# Calendário contínuo até hoje (se a última cotação for de sexta e hoje for domingo, o fim de semana é preenchido)
fim_serie = max(limites["fim"], date.today())
calendario = spark.range(1).select(
    F.explode(F.sequence(F.lit(limites["ini"]), F.lit(fim_serie))).alias("data_cotacao")
)

# Forward Fill: last(..., ignorenulls=True) sobre a janela acumulada ordenada por data
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
silver_cotacao = (
    calendario.join(dias_uteis, "data_cotacao", "left")
    .withColumn("preenchida_forward_fill", F.col("cotacao_dolar").isNull())
    .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(w_ff))
    .select("data_cotacao", "cotacao_dolar", "preenchida_forward_fill")
    .orderBy("data_cotacao")
)

write_silver(silver_cotacao, "silver.tb_cotacao_dolar")
display(spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).limit(15))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[ok] silver.tb_cotacao_dolar: 8 linhas


data_cotacao,cotacao_dolar,preenchida_forward_fill
2026-09-21,5.111100,false
2026-09-20,5.156900,true
2026-09-19,5.156900,true
2026-09-18,5.156900,false
2026-09-17,5.151500,false
2026-09-16,5.152000,false
2026-09-15,5.148400,false
2026-09-14,5.169000,false


Regras: deduplicação por filme (ingestão mais recente), status normalizado e traduzido, data lançada por múltiplos formatos, `ano_lancamento` derivado.

In [0]:
info = latest_version(prepare_id(spark.table("bronze.tb_movies_info")))
require_columns(
    info,
    ["id", "title", "original_title", "release_date", "runtime", "original_language", "status", "overview", "tagline"],
    "bronze.tb_movies_info",
)

# --- 2.1 Status: normaliza (remove ruídos/hífens/underscores, caixa) e só depois traduz ---------
status_norm = F.lower(
    F.trim(F.regexp_replace(F.regexp_replace(F.col("status").cast("string"), r"[^A-Za-z]+", " "), r"\s+", " "))
)
MAPA_STATUS = {
    "released": "Lançado",
    "post production": "Pós-Produção", "postproduction": "Pós-Produção",
    "in production": "Em Produção", "inproduction": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores", "rumoured": "Rumores",
    "canceled": "Cancelado", "cancelled": "Cancelado",
}
# Encadeamento de when() em vez de map[...]: no modo ANSI, chave inexistente em map lança erro
status_filme = None
for chave, traducao in MAPA_STATUS.items():
    status_filme = F.when(status_norm == chave, traducao) if status_filme is None else status_filme.when(status_norm == chave, traducao)
status_filme = status_filme.otherwise("Não Informado")   # corrompido / não mapeável

# --- 2.2 Data de lançamento multi-formato ------------------------------------------------------
# Remove horário no final (ex.: '2020-01-05 00:00:00', '01/05/2020 10:30 AM') antes de testar os formatos
dt_txt = F.regexp_replace(
    F.trim(F.col("release_date").cast("string")),
    r"[ T]\d{1,2}:\d{2}(:\d{2}(\.\d+)?)?\s?([APap][Mm])?Z?$",
    "",
)


def try_fmt(fmt):
    """Tenta converter no formato; devolve NULL (sem erro) se não casar."""
    return F.try_to_timestamp(dt_txt, F.lit(fmt)).cast("date")


# Ambiguidade dd/MM vs MM/dd: decidida pelos DADOS. Conta datas que só fazem sentido em cada ordem
# (ex.: 25/12/2020 só existe como dia-primeiro; 12/25/2020 só como mês-primeiro).
n_dia_primeiro = info.filter(try_fmt("d/M/yyyy").isNotNull() & try_fmt("M/d/yyyy").isNull()).count()
n_mes_primeiro = info.filter(try_fmt("M/d/yyyy").isNotNull() & try_fmt("d/M/yyyy").isNull()).count()
DIA_PRIMEIRO = n_dia_primeiro >= n_mes_primeiro   # empate/sem evidência -> padrão brasileiro
print(f"Datas inequívocas dd/MM: {n_dia_primeiro} | MM/dd: {n_mes_primeiro} -> dia primeiro = {DIA_PRIMEIRO}")

fmts_barra = ["d/M/yyyy", "M/d/yyyy"] if DIA_PRIMEIRO else ["M/d/yyyy", "d/M/yyyy"]
fmts_hifen = ["d-M-yyyy", "M-d-yyyy"] if DIA_PRIMEIRO else ["M-d-yyyy", "d-M-yyyy"]
FORMATOS = (
    ["yyyy-M-d", "yyyy/M/d"]                 # ISO e variações
    + fmts_barra + fmts_hifen                # 05/06/2020 e 05-06-2020
    + ["d.M.yyyy"]                           # 05.06.2020
    + ["MMM d, yyyy", "MMMM d, yyyy", "d MMM yyyy", "d MMMM yyyy", "MMM d yyyy", "MMMM d yyyy"]   # 'Jan 5, 2020', '5 January 2020'
)
candidatos = [try_fmt(f) for f in FORMATOS]
candidatos.append(F.when(dt_txt.rlike(r"^\d{8}$"), try_fmt("yyyyMMdd")))   # 20200105
candidatos.append(F.when(dt_txt.rlike(r"^\d{4}$"), try_fmt("yyyy")))       # só o ano -> 01/01 (último recurso)
data_lancamento = F.coalesce(*candidatos)   # 1º formato que funcionar; NULL só se nenhum servir

# Auditoria: o que ficou sem conversão? (se houver um padrão recorrente, inclua-o em FORMATOS)
nao_convertidas = (
    info.filter(dt_txt.isNotNull() & (dt_txt != "") & data_lancamento.isNull())
    .groupBy(dt_txt.alias("valor_original")).count().orderBy(F.desc("count"))
)
print("Valores de data que ficaram NULL (amostra):")
display(nao_convertidas.limit(20))

# --- 2.3 Montagem da tabela --------------------------------------------------------------------
duracao = to_int(F.col("runtime"))
silver_info = (
    info.select(
        F.col("id").alias("id_filme"),
        blank_to_null(F.col("title")).alias("titulo"),
        blank_to_null(F.col("original_title")).alias("titulo_original"),
        data_lancamento.alias("data_lancamento"),
        F.when(duracao > 0, duracao).alias("duracao_minutos"),   # duração <= 0 não é válida
        F.lower(blank_to_null(F.col("original_language"))).alias("idioma_original"),
        status_filme.alias("status_filme"),
        blank_to_null(F.col("overview")).alias("sinopse"),
        blank_to_null(F.col("tagline")).alias("frase_divulgacao"),
    )
    .withColumn("ano_lancamento", F.year("data_lancamento"))
)

write_silver(silver_info, "silver.tb_info_filmes")
print("Distribuição do status traduzido:")
display(spark.table("silver.tb_info_filmes").groupBy("status_filme").count().orderBy(F.desc("count")))

Datas inequívocas dd/MM: 5639 | MM/dd: 0 -> dia primeiro = True
Valores de data que ficaram NULL (amostra):


valor_original,count
"Be Not Proud: The Making of \""\""The Exorcist III\""\""\""""",1
"You're Dead! The Making of \""\""House\""\""\""""",1


[ok] silver.tb_info_filmes: 97,879 linhas
Distribuição do status traduzido:


status_filme,count
Lançado,96522
Pós-Produção,701
Em Produção,604
Planejado,47
Não Informado,5


Regras: ausência textual → NULL; remoção de símbolos de moeda e pontuação de milhar; `DECIMAL(18,2)`; zero/negativo → NULL;
conversão para BRL pela cotação mais recente da série; lucro e margem.

**Decisão sobre o lucro (comentário de negócio):** o escopo pede que operações com valores ausentes "não invalidem o resultado".
Com `LUCRO_AUSENTE_COMO_ZERO = True`, um dos lados ausente é tratado como 0 no cálculo (receita sem orçamento → lucro = receita).
Se **ambos** forem ausentes o lucro permanece NULL (não inventamos um 0). Mude para `False` para exigir os dois valores.

In [0]:
LUCRO_AUSENTE_COMO_ZERO = True

fin = latest_version(prepare_id(spark.table("bronze.tb_movies_financials")))
require_columns(fin, ["id", "budget", "revenue"], "bronze.tb_movies_financials")

# Cotação: a mais recente da série contínua (filmes não têm data de transação)
linha_taxa = spark.table("silver.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()
assert linha_taxa is not None, "silver.tb_cotacao_dolar vazia"
print(f"Cotação aplicada: R$ {linha_taxa['cotacao_dolar']} (data {linha_taxa['data_cotacao']})")
taxa = F.lit(str(linha_taxa["cotacao_dolar"])).cast("decimal(18,6)")


def money(c):
    """Texto -> DECIMAL(18,2): ausência textual e valores <= 0 viram NULL."""
    texto = c.cast("string")
    sem_ausente = F.when(F.lower(F.trim(texto)).isin(AUSENTES), F.lit(None)).otherwise(texto)
    valor = to_decimal(sem_ausente, 18, 2, single_sep_is_decimal=False, allow_currency=True)
    return F.when(valor > 0, valor)


def brl(usd):
    return (usd * taxa).cast("decimal(18,2)")


def diferenca(a, b):
    """a - b tratando ausentes conforme a decisão de negócio acima."""
    if LUCRO_AUSENTE_COMO_ZERO:
        return F.when(a.isNull() & b.isNull(), F.lit(None)).otherwise(
            F.coalesce(a, F.lit(0)) - F.coalesce(b, F.lit(0))
        )
    return a - b


silver_fin = (
    fin.select(
        F.col("id").alias("id_filme"),
        money(F.col("budget")).alias("orcamento_usd"),
        money(F.col("revenue")).alias("receita_usd"),
    )
    .withColumn("orcamento_brl", brl(F.col("orcamento_usd")))
    .withColumn("receita_brl", brl(F.col("receita_usd")))
    .withColumn("lucro_usd", diferenca(F.col("receita_usd"), F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", diferenca(F.col("receita_brl"), F.col("orcamento_brl")).cast("decimal(18,2)"))
    # Margem % = lucro / receita * 100; receita nula ou <= 0 -> NULL (evita divisão por zero)
    .withColumn(
        "margem_lucro_pct",
        F.when(
            F.col("receita_usd") > 0,
            F.round(F.col("lucro_usd").cast("double") / F.col("receita_usd").cast("double") * 100, 2),
        ),
    )
)

write_silver(silver_fin, "silver.tb_financeiro_filmes")
display(spark.table("silver.tb_financeiro_filmes").orderBy(F.col("receita_usd").desc_nulls_last()).limit(10))

Cotação aplicada: R$ 5.111100 (data 2026-09-21)
[ok] silver.tb_financeiro_filmes: 99,006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_pct
299534,356000000.00,2800000000.00,1819551600.00,14311080000.00,2444000000.00,12491528400.00,87.29
76600,460000000.00,2320250281.00,2351106000.00,11859031211.22,1860250281.00,9507925211.22,80.17
299536,300000000.00,2052415039.00,1533330000.00,10490098505.83,1752415039.00,8956768505.83,85.38
634649,200000000.00,1921847111.00,1022220000.00,9822752769.03,1721847111.00,8800532769.03,89.59
420818,260000000.00,1663075401.00,1328886000.00,8500144682.05,1403075401.00,7171258682.05,84.37
361743,170000000.00,1488732821.00,868887000.00,7609062321.41,1318732821.00,6740175321.41,88.58
346698,null,1428545028.00,null,7301436492.61,1428545028.00,7301436492.61,100.0
502356,100000000.00,1355725263.00,511110000.00,6929247391.72,1255725263.00,6418137391.72,92.62
284054,200000000.00,1349926083.00,1022220000.00,6899607202.82,1149926083.00,5877387202.82,85.18
181808,200000000.00,1332698830.00,1022220000.00,6811556990.01,1132698830.00,5789336990.01,84.99


Regras: popularidade com pontuação/decimal inconsistentes é higienizada antes do cast; textos do *Column Shift* → NULL (sem quebrar o pipeline);
notas fora de [0, 10] (inclusive escala multiplicada) → NULL; votos e popularidade negativos → NULL.

In [0]:
met = latest_version(prepare_id(spark.table("bronze.tb_movies_metrics")))
require_columns(met, ["id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes"], "bronze.tb_movies_metrics")


def nota_valida(c):
    """Nota 0-10; fora do intervalo (ex.: 75 por erro de escala) é descartada."""
    v = to_double(c, single_sep_is_decimal=True)
    return F.when((v >= 0) & (v <= 10), v)


def votos_validos(c):
    """Contagem de votos: inteiro >= 0; texto/negativo -> NULL."""
    v = to_int(c, single_sep_is_decimal=False)
    return F.when(v >= 0, v)


popularidade = to_double(F.col("popularity"), single_sep_is_decimal=True)   # '12,345' -> 12.345

silver_met = met.select(
    F.col("id").alias("id_filme"),
    F.when(popularidade >= 0, popularidade).alias("popularidade"),
    nota_valida(F.col("vote_average")).alias("nota_media_tmdb"),
    votos_validos(F.col("vote_count")).alias("qtd_votos_tmdb"),
    nota_valida(F.col("averageRating")).alias("nota_media_imdb"),
    votos_validos(F.col("numVotes")).alias("qtd_votos_imdb"),
)

write_silver(silver_met, "silver.tb_metricas_engajamento")
display(spark.table("silver.tb_metricas_engajamento").limit(10))

[ok] silver.tb_metricas_engajamento: 99,013 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.132,0.0,0,6.8,27
1000005,0.6,0.0,0,null,40
1000007,0.6,0.0,0,4.7,10
1000011,1.169,0.0,0,4.8,16
1000014,0.6,0.0,0,7.2,15
1000030,0.615,0.0,0,null,25
1000054,0.6,0.0,0,5.9,10
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236


Regras: duplicatas integrais removidas; nota fora de 0–10 → NULL; comentário vazio/só espaços → "Sem comentário".

In [0]:
rev = prepare_id(spark.table("bronze.tb_movies_reviews"))
require_columns(rev, ["id", "nome", "nota", "comentario"], "bronze.tb_movies_reviews")

nota = to_double(F.col("nota"), single_sep_is_decimal=True)

silver_rev = (
    rev.select(
        F.col("id").alias("id_filme"),
        blank_to_null(F.col("nome")).alias("nome_usuario"),
        F.when((nota >= 0) & (nota <= 10), nota).alias("nota_usuario"),
        F.coalesce(blank_to_null(F.col("comentario")), F.lit("Sem comentário")).alias("comentario_usuario"),
    )
    # Duplicata integral = mesma combinação filme + usuário + nota + comentário (já normalizados)
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

write_silver(silver_rev, "silver.tb_avaliacoes_usuarios")
display(spark.table("silver.tb_avaliacoes_usuarios").limit(10))

[ok] silver.tb_avaliacoes_usuarios: 32,412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


Regras: `split` por `,`, `;` **ou** `|` → `explode` → `trim`. Como a coluna tem lixo (Column Shift, textos descritivos, números),
só permanecem valores do **domínio de gêneros** (lista TMDB + IMDb, comparação sem diferença de caixa). Caso eu queira aceitar algum gênero que não está sendo reconhecido, posso incluí-lo em 'GENEROS_EXTRA'.

In [0]:
cred = latest_version(prepare_id(spark.table("bronze.tb_credits_and_tags")))
require_columns(cred, ["id", "genres", "cast", "directors", "writers", "production_companies"], "bronze.tb_credits_and_tags")

GENEROS_VALIDOS = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama", "Family", "Fantasy",
    "History", "Horror", "Music", "Musical", "Mystery", "Romance", "Science Fiction", "TV Movie",
    "Thriller", "War", "Western", "Biography", "Film-Noir", "Sport", "Short", "News", "Reality-TV",
    "Talk-Show", "Game-Show", "Adult",
]
GENEROS_EXTRA = []   # acrescente aqui gêneros legítimos que apareçam na auditoria abaixo
ALIAS_GENEROS = {"sci-fi": "Science Fiction", "scifi": "Science Fiction", "science-fiction": "Science Fiction"}

linhas_dominio = {g.lower(): g for g in GENEROS_VALIDOS + GENEROS_EXTRA}
linhas_dominio.update({k: v for k, v in ALIAS_GENEROS.items()})
dominio_generos = spark.createDataFrame(list(linhas_dominio.items()), "chave STRING, genero STRING")

tokens_generos = (
    cred.select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.col("genres").cast("string"), r"[,;|]")).alias("token"),   # trata vírgula, ponto e vírgula e barra vertical
    )
    # remove colchetes/aspas residuais, espaços duplicados e padroniza a caixa para comparar com o domínio
    .withColumn("chave", F.lower(F.trim(F.regexp_replace(F.regexp_replace("token", r"""[\[\]'"]""", ""), r"\s+", " "))))
)

silver_generos = (
    tokens_generos.join(F.broadcast(dominio_generos), "chave", "inner")
    .select("id_filme", "genero")
    .distinct()
)

write_silver(silver_generos, "silver.tb_generos")

print("Auditoria - valores DESCARTADOS (mais frequentes):")
display(
    tokens_generos.join(F.broadcast(dominio_generos), "chave", "left_anti")
    .filter(F.col("chave") != "")
    .groupBy("chave").count().orderBy(F.desc("count")).limit(30)
)

[ok] silver.tb_generos: 142,551 linhas
Auditoria - valores DESCARTADOS (mais frequentes):


chave,count
0.6,267
united states of america,32
1.4,26
english,19
0.0,18
nenhum,9
n/a,9
france,8
japan,8
germany,7


Dimensão unificada: `cast`→Ator, `directors`→Diretor, `writers`→Roteirista, `production_companies`→Produtora.
Regras: split por `,`/`;`/`|` + `posexplode` (guarda `ordem_credito` para identificar os atores principais), capitalização padronizada (`initcap`),
remoção de lixo e deduplicação por filme + nome + tipo.

Filtro de lixo (heurística): descarta nulos/vazios, textos sem nenhuma letra (números/datas deslocados), textos muito longos
(descrições), termos de ausência e valores que são gêneros (Column Shift).

In [0]:
FONTES = [("cast", "Ator"), ("directors", "Diretor"), ("writers", "Roteirista"), ("production_companies", "Produtora")]

partes = []
for coluna, tipo in FONTES:
    partes.append(
        cred.select(
            F.col("id").alias("id_filme"),
            F.posexplode(F.split(F.col(coluna).cast("string"), r"[,;|]")).alias("ordem_credito", "token"),
        )
        .withColumn("nome_entidade", F.initcap(F.regexp_replace(F.trim(F.col("token")), r"\s+", " ")))
        .withColumn("tipo_entidade", F.lit(tipo))
        .select("id_filme", "nome_entidade", "tipo_entidade", "ordem_credito")
    )

bruto = partes[0]
for p in partes[1:]:
    bruto = bruto.unionByName(p)

nome = F.col("nome_entidade")
entidade_valida = (
    nome.isNotNull()
    & (F.length(nome) >= 2)
    & (F.length(nome) <= 70)                              # textos descritivos são longos
    & (F.size(F.split(nome, " ")) <= 8)
    & nome.rlike(r"\p{L}")                                # precisa ter letra: descarta números/datas deslocados
    & ~F.lower(nome).isin(AUSENTES)
    & ~F.lower(nome).isin(list(linhas_dominio.keys()))    # gênero dentro de cast/diretor/... = Column Shift
)

silver_pessoas = (
    bruto.filter(entidade_valida)
    .groupBy("id_filme", "nome_entidade", "tipo_entidade")    # elimina duplicatas mantendo a melhor posição no crédito
    .agg(F.min("ordem_credito").alias("ordem_credito"))
)

write_silver(silver_pessoas, "silver.tb_pessoas_empresas")
display(spark.table("silver.tb_pessoas_empresas").groupBy("tipo_entidade").count())

[ok] silver.tb_pessoas_empresas: 891,682 linhas


tipo_entidade,count
Ator,544131
Diretor,102180
Roteirista,126668
Produtora,118703


In [0]:
TABELAS_SILVER = [
    "silver.tb_cotacao_dolar", "silver.tb_info_filmes", "silver.tb_financeiro_filmes",
    "silver.tb_metricas_engajamento", "silver.tb_avaliacoes_usuarios", "silver.tb_generos",
    "silver.tb_pessoas_empresas",
]
for t in TABELAS_SILVER:
    print(f"{t:<36} {spark.table(t).count():>10,} linhas")

# Unicidade por filme nas tabelas de grão "1 linha por filme"
for t in ["silver.tb_info_filmes", "silver.tb_financeiro_filmes", "silver.tb_metricas_engajamento"]:
    df = spark.table(t)
    assert df.count() == df.select("id_filme").distinct().count(), f"{t} tem id_filme duplicado"
print("Unicidade por filme: OK")

spark.table("silver.tb_info_filmes").printSchema()
spark.table("silver.tb_financeiro_filmes").printSchema()
spark.table("silver.tb_metricas_engajamento").printSchema()

silver.tb_cotacao_dolar                       8 linhas
silver.tb_info_filmes                    97,879 linhas
silver.tb_financeiro_filmes              99,006 linhas
silver.tb_metricas_engajamento           99,013 linhas
silver.tb_avaliacoes_usuarios            32,412 linhas
silver.tb_generos                       142,551 linhas
silver.tb_pessoas_empresas              891,682 linhas
Unicidade por filme: OK
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)

root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = t